# PRÁCTICA PR02 (PARTE GUIADA) | PROBABILIDAD PRyES
## Trabajo con temperaturas mínimas y máximas de España (fuente: AEMET)
Objetivos de aprendizaje
- Determinar probabilidades a partir de datos reales
- Calcular probabilidades a partir de las distribuciones de probabilidad estimadas
- Relacionar los modelos creados con los datos reales disponibles
- Manejar el conjunto de datos usando un dataframe de pandas
- Sentar las bases para el trabajo en equipo sobre estadística descriptiva e inferencia

### 0: Carga de librerías necesarias

In [ ]:
# Importar librerías
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

### 1: Carga de datos

In [ ]:
# leemos los datos del fichero con las temperaturas usando pandas
# Los datos están en ºC

# df = pd.read_csv('TemperaturesSpain.csv', index_col=['YEAR','MONTH','DATE'])
df_orig = pd.read_csv('TemperaturesSpain.csv')

# parseamos la fecha (cadena) para que sea un datetime con formato dd/mm/yyyy
df_orig["DATE"] = pd.to_datetime(df_orig["DATE"], format='%d/%m/%Y') 
print('Tamaño de df_orig con los datos cargados:', df_orig.shape)
print('Dataframe con todos los datos leidos (filas: días, columnas: variables temperaturas):')
df_orig

# 1. Análisis probabilista del comportamiento de la serie de temperaturas TMAX229

### 1.1. ¿Qué probabilidad hay de que TMAX229 de mañana sea inferior a la TMAX229 de hoy?

Asignar una probabilidad a partir de los datos disponibles, siempre que tenga sentido ...

In [ ]:
# Conjetura incial 
print( '... humm ... probabilidad = 50% ?')

# Veamos lo que ha pasado históricamente

# 1º calculamos la diferencia entre todos los T_d+1 (mañana)
# y T_d (hoy):  dif = T_d+1 - T_d
df_dif = df_orig.loc[:,['DATE', 'MONTH', 'TMAX229']]
df_dif['TMAX229_tomorrow'] = df_dif.TMAX229.shift(periods = -1)
df_dif['DIF'] = df_dif.TMAX229_tomorrow - df_dif.TMAX229 
print('DF con las diferencias diarias entre hoy y mañana:\n',df_dif)

# chequeamos que lo hemos planteado bien
fig = plt.figure(1,figsize=(15,4))
plt.plot(df_dif.DATE[0:30], df_dif.TMAX229[0:30], label= 'TMAX229')
plt.plot(df_dif.DATE[0:30], df_dif.TMAX229_tomorrow[0:30], label= 'TMAX229_TOMORROW')
plt.stem(df_dif.DATE[0:30], df_dif.DIF[0:30], label= 'DIF')
plt.title('Temperatura hoy y mañana y su diferencia (primeros 30 días)')
plt.legend()
plt.show()

In [ ]:
# 2º estudiamos cuando dif < 0, i.e. cuando la temperatura de
# mañana es inferior a la de hoy

# contamos cuantas veces dif<0

es_inferior = df_dif.DIF<0
print('Variable que indica cuando es inferior a cero:\n', es_inferior)
num_dias_es_inferior = sum(es_inferior)
num_dias_total = df_dif.shape[0]
print('Número de días que es inferior:', num_dias_es_inferior)
print('Número de días total:', num_dias_total)
freq_dias_inferior = num_dias_es_inferior / num_dias_total
p_baje_TMAX229 = freq_dias_inferior
print('Frecuencia dias que es inferior:', np.round(freq_dias_inferior*100,2), '%')
print('Entonces la probabilidad de que la TMAX229 de mañana sea inferior a la de hoy no es 50%, es inferior (',  np.round(p_baje_TMAX229,4),')')
print('¿Por qué no se aproxima más al 50%? ¿Cambio climático?')


# observamos la serie de diferencias completa
fig = plt.figure(1,figsize=(15,4))

plt.stem(df_dif.DATE, df_dif.DIF, label= 'DIF')
plt.title('Diferencia entre temperaturas hoy y mañana (todos los años)')
plt.legend()
plt.show()

### 1.2. ¿En qué mes del año es menor la probabilidad de que la TMAX229 de mañana sea inferior a la TMAX229 de hoy?

Asignar una probabilidad a partir de los datos disponibles, probabilidad condicionada

In [ ]:
# Repetimos el cálculo de las diferencias diarias y calculamos para cada mes

# Usamos el dataframe df_dif, creando una columna de diferencias para cada mes
df_dif = df_orig.loc[:,['DATE', 'MONTH', 'TMAX229']]
df_dif['TMAX229_tomorrow'] = df_dif.TMAX229.shift(periods = -1)
df_dif['DIF'] = df_dif.TMAX229_tomorrow - df_dif.TMAX229 
print('DF con las diferencias diarias entre hoy y mañana:\n',df_dif)

# desapilamos el df original y nos quedamos con las columnas de diferencias mensuales
df_dif_mes = df_dif.pivot(index='DATE', columns='MONTH')

df = df_dif_mes['DIF']
df.columns = ['DM1','DM2','DM3','DM4','DM5','DM6','DM7','DM8','DM9','DM10', 'DM11', 'DM12']
print('DF con las diferencias diarias entre hoy y mañana organizadas por MESES:\n', df)

In [ ]:
# representamos un par de columnas para verificar y entender
fig = plt.figure(1,figsize=(15,4))
df.DM1.plot(label='Dif enero')
df.DM5.plot(label='Dif mayo')
plt.title('Diferencias diarias enero y mayo')
plt.legend()
plt.show()

# Visión general a las diferencias mensuales
fig = plt.figure(2,figsize=(15,4))
df.boxplot(rot=45)
plt.title('Distribución de las diferencias según los meses')
plt.show()
print('Las diferencias varían según los meses, ¿Por qué?')

# Representamos lo que pasa según los meses
fig = plt.figure(3,figsize=(15,4))
plt.plot(df_orig[df_orig.YEAR == 2015].TMAX229.tolist(), label='2015')
plt.plot(df_orig[df_orig.YEAR == 2010].TMAX229.tolist(), label='2010')
plt.plot(df_orig[df_orig.YEAR == 1990].TMAX229.tolist(), label='1990')
plt.title('Evolución diaria de TMAX229 en 3 años diferentes')
plt.grid()
plt.legend()
plt.show()

In [ ]:
# Estimamos probabilidades a partir de las frecuencias mensuales

# nº de días en cada mes
df_num_dias_mes = df.count(0)
# Nº de días en cada mes en los que la diferencia es negativa
df_num_dias_inferior_mes = df[df<0].count()
# Organizamos todo en un df
df_mes = pd.concat([df_num_dias_mes, df_num_dias_inferior_mes], axis=1)
df_mes.columns = ['NUM_DIAS_TOTAL', 'NUM_DIAS_INFERIOR']
df_mes['FRECUENCIA'] = df_mes.NUM_DIAS_INFERIOR / df_mes.NUM_DIAS_TOTAL
print('Nº de días, nº con dif<0 y frecuencias para cada mes\n', df_mes)


# Visión general a las diferencias mensuales
fig = plt.figure(2,figsize=(10,4))
plt.bar(df_mes.index, df_mes.FRECUENCIA)
plt.grid()
plt.title('Probabilidad ( TMAX229(d+1)<TMAX229(d)  | mes )')
plt.show()

# RESPUESTA
print('RESPUESTA: Es MAYO el mes donde es menor la probabilidad de que la TMAX229 de mañana sea inferior a la TMAX229 de hoy')


### 1.3. Sabiendo que la temperatura TMAX229 fue inferior a la del día anterior ¿Qué probabilidad hay de estar en Agosto?

Cálculo de una probabilidad condicionada, teorema de Bayes

In [ ]:
# Cálculo a partir de la partición según meses anterior

print('Hay que calcular P( agosto | TMAX229(d+1)<TMAX229(d) ) = P(agosto | BAJA)')
print('P(agosto | BAJA) = P(agosto & BAJA) / P(BAJA)')

# denominador P(BAJA) ya la hemos calculado
p_BAJA = p_baje_TMAX229
print('La probabilidad de que sea TMAX229(d+1)<TMAX229(d):', p_BAJA)

# numerador P(agosto & BAJA)
print('Usamos la partición para obtener P(agosto & BAJA) = P(BAJA | agosto) * P(agosto)')

# la probabilidad de agosto a partir de los datos que tenemos :)
num_dias_total = df_mes.NUM_DIAS_TOTAL.sum()
num_dias_agosto = df_mes.loc['DM8', 'NUM_DIAS_TOTAL']
print('  - Nº total días:', num_dias_total, '. Nº total agosto:', num_dias_agosto)
p_agosto = num_dias_agosto/num_dias_total
print('  - P(agosto)=', p_agosto)

# P(BAJA | agosto) la hemos calculado antes (regla de Laplace)
df_mes8 = df_mes.iloc[[7],:]
print(df_mes8)
p_BAJA_sabiendoAgosto = df_mes8.FRECUENCIA[0]
print('  - P(BAJA | agosto) =', p_BAJA_sabiendoAgosto)
print('  - P(agosto & BAJA) = P(BAJA | agosto) * P(agosto)=', p_BAJA_sabiendoAgosto * p_agosto)
print('La probabilidad pedida: '
      'P(agosto | BAJA) = P(agosto & BAJA) / P(BAJA)=', p_BAJA_sabiendoAgosto * p_agosto / p_BAJA)

print('Se cumple que P(agosto) =~ P(agosto | BAJA), es decir, saber que la temperatura bajó '
      'no ayuda demasiado a determinar si es o no agosto (lógico)' )


### 1.4. ¿Cuál es la probabilidad de que baje 40 días la TMAX229 respecto al día anterior en 100 días seguidos?¿Y que baje como poco 50 días de los 100 días seguidos?¿Cuántos días de los 100 bajará de media?

Uso de distribuciones discretas

In [ ]:
# BINOMIAL

print('Cada día puede bajar o no bajar. La probabilidad de que baje (éxito) es: p=', p_baje_TMAX229)
print('Podemos asumir que dicha probabilidad p es constante y que se trata de un secuencia de experimientos de Bernoulli independientes')

print('Sea X1: v.a. nº de días que baja TMAX229 en 100 días seguidos')
print('X1 ~ Binomial(n=100, p = ',  p_baje_TMAX229,')')

# Creamos la binomial X1 que lo modela y operamos con ella
n= 100
p= p_baje_TMAX229
X1 = stats.binom(n=n,p=p)

# Probabilidades pedidas
p_baje_40dias = X1.pmf(40)
p_baje_mas50dias = 1 - X1.cdf(49)
print('La probabilidad de que TMAX229 baje exactamente 40 días es:', p_baje_40dias)
print('La probabilidad de que TMAX229 baje al menos 50 días es:', p_baje_mas50dias)
# esperanza pedida
num_dias_medio_baja = X1.mean()
print('El nº de días esperado en 100 días que TMAX229 baja es:', num_dias_medio_baja)

# Visualmente
xs = range(30, 60)
fig = plt.figure(1, figsize=(10, 4))
ax1 = plt.subplot(2,1,1)
plt.stem(xs, X1.pmf(xs), basefmt='none')
plt.ylabel('pmf(x)')
plt.grid()
plt.title('Binomial(n='+ str(n)+', p='+ str(round(p,4))+')')
plt.xlim(xs[0], xs[-1])
plt.subplot(2,1,2, sharex = ax1)
plt.step(xs, X1.cdf(xs), color='red', where='post')
plt.xlabel('x')
plt.ylabel('cdf(x)')
plt.grid()
plt.show()

In [ ]:
# POISSON (otra posible opción razonable)

print('Cada día puede bajar o no bajar, pero no pueden pasar ambas cosas.')
print('Se puede asumir una tasa media de bajada constante: lambda=', p_baje_TMAX229 * 100, 'bajadas por cada 100 días')
print('Además podemos asumir que las bajadas son independientes')

print('Sea X2: v.a. nº de días que baja TMAX229 en 100 días seguidos')
print('X2 ~ Poisson(lambda=',  p_baje_TMAX229 * 100,')')

# Creamos la poisson X2 que lo modela y operamos con ella
lmda= p_baje_TMAX229 * 100
X2 = stats.poisson(lmda)

# Probabilidades pedidas
p_baje_40dias = X2.pmf(40)
p_baje_mas50dias = 1 - X2.cdf(49)
print('La probabilidad de que TMAX229 baje exactamente 40 días es:', p_baje_40dias)
print('La probabilidad de que TMAX229 baje al menos 50 días es:', p_baje_mas50dias)
# esperanza pedida
num_dias_medio_baja = X2.mean()
print('El nº de días esperado en 100 días que TMAX229 baja es:', num_dias_medio_baja)

# Visualmente
xs = range(30, 60)
fig = plt.figure(1, figsize=(10, 4))
ax1 = plt.subplot(2,1,1)
plt.stem(xs, X2.pmf(xs), basefmt='none')
plt.ylabel('pmf(x)')
plt.grid()
plt.title('Poisson(lambda='+ str(lmda)+')')

plt.subplot(2,1,2, sharex = ax1)
plt.xlim(xs[0], xs[-1])
plt.step(xs, X2.cdf(xs), color='red', where='post')
plt.xlabel('x')
plt.ylabel('cdf(x)')
plt.grid()
plt.show()

In [ ]:
# Comparamos con lo que está pasando en la realidad (**)

# cogemos conjuntos aleatorios de 100 días seguidos y contamos cuantas veces TMAX229 bajó
df_dif_base = df_dif.loc[:,['DIF']]
print('nº de filas:',  df_dif_base.shape[0])

print('Creamos num_series seleccionadas aleatoriamente de la serie original de diferencias')
tam = 100
num_series = 1000
np.random.seed = 9999
df_dif_random_series = pd.DataFrame()
for i in range(0, num_series):
    ini = np.random.randint(1, df_dif_base.shape[0]-tam-1)
    df_dif_base_shifted = df_dif_base.DIF.shift(periods = - ini)
    df_dif_base_shifted_primeros_tam_nuevo = df_dif_base_shifted.iloc[0:tam]
    df_dif_random_series = pd.concat([df_dif_random_series, df_dif_base_shifted_primeros_tam_nuevo], axis=1)

print(df_dif_random_series)

# chequeamos visualmente
df_dif_random_series.plot(figsize=(15, 4), legend='')

In [ ]:
# Contamos en cada secuencia (columna DIF del df) cuantos negativos hay
df_num_dias_bajadas_seq = df_dif_random_series[df_dif_random_series<0].count()
num_dias = df_num_dias_bajadas_seq.tolist()
print('Valores calculados a partir de la muestra real:', num_dias[0:20], '...')

xs = range(30, 60)
# pintamos histograma con frecuencias y asegurando que la suma de sus barras es 1 (como una pmf)
hist, bins = np.histogram(num_dias, bins=120)
fig = plt.figure(1, figsize=(15, 6))
plt.bar(bins[:-1], hist.astype(np.float32) / hist.sum(), width=(bins[1]-bins[0]), color='red', label='Histograma real')
plt.stem(xs, X1.pmf(xs), basefmt='none')
plt.plot(xs, X1.pmf(xs),  'k', label='Binomial')
plt.stem(xs, X2.pmf(xs), basefmt='none')
plt.plot(xs, X2.pmf(xs),  'g', label='Poisson')
plt.xlabel('Nº de bajadas en 100 días seguidos')
plt.legend()
plt.show()

# 2. Modelado de los errores de estimación de una IA naive

Vamos a estimar la temperatura TMAX229 de mañana a partir de la temperatura de hoy de una forma muy muy simplona:

y_est(d+1) = y(d)

Definimos el error de estimación como la diferencia entre el real y el estimado: e(d) = y(d) - y_est(d)

In [ ]:
# se parece bastante al análisis inicial

# 1º estimamos para cada día a partir del dato anterior y calculamos errores
df_IA = df_orig.loc[:,['DATE', 'MONTH', 'TMAX229']]
df_IA['TMAX229_est'] =  df_IA.TMAX229.shift(periods = +1)
df_IA['ERROR'] = df_IA.TMAX229 - df_IA.TMAX229_est 
# quitamos los nan
df_IA = df_IA.dropna()

print('DF de la IA naive:\n',df_IA)

# chequeamos que lo hemos planteado bien
fig = plt.figure(1,figsize=(15,5))
plt.plot(df_IA.DATE[0:100], df_IA.TMAX229[0:100],'.-', label= 'TMAX229')
plt.plot(df_IA.DATE[0:100], df_IA.TMAX229_est[0:100],'.-', label= 'TMAX229_est')
plt.stem(df_IA.DATE[0:100], df_IA.ERROR[0:100], label= 'ERROR')
plt.title('Temperatura real y estimada y error cometido (primeros 30 días)')
plt.legend()
plt.show()

### 2.1. Calcular la probabilidad de que el error comentido por la IA esté entre -2ºC y +2ºC

In [ ]:
print('Podemos pensar que los errores diarios se distribuyen como una Normal')
print('Si la IA tiene algo de I, entonces la media debería ser nula')
print('La desviación típica no es fácil de suponer, pero la podemos estimar a partir de la muestra que tenemos\n')

print('Sea E: v.a. error diario de predicción de la IA')
print('E ~ Normal(mu, sigma)')

# calculamos la media y la desviación típica de los errores que tenemos
media_est = np.mean(df_IA.ERROR)
sigma_est = np.std(df_IA.ERROR)

print('Entoces podemos suponer que E ~ Normal(mu=', media_est, ', sigma =', sigma_est,')')

# creamos la variable y operamos
mu = media_est
sigma = sigma_est
E = stats.norm(loc = mu, scale = sigma)

p_error_mayor_2grados_pos = E.cdf(2)
p_error_menor_2grados_neg = E.cdf(-2)
p_error_entre_2neg_2pos = p_error_mayor_2grados_pos - p_error_menor_2grados_neg

print('P( -2ºC <  error comentido por la IA < +2ºC) =', p_error_entre_2neg_2pos)


In [ ]:
# representamos la distribución real de los errores y la distribución Normal estimada
errores = df_IA.ERROR.tolist()
errores
xs = np.linspace(-10, 10, 1000)

# pintamos histograma en densidad (freq. relativa)
fig = plt.figure(1, figsize=(15, 6))
plt.hist(errores, bins=120, density=True, label='Histograma errores reales')
plt.plot(xs, E.pdf(xs),  'k', label='Normal estimada')
plt.xlabel('Error IA (ºC)')
plt.legend()
plt.show()

print('¿Es una normal? ¿Qué diferencias importantes se observan?')